In [79]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import pdal as pdal


In [80]:
def load_las_as_numpy(las_path):
    pipeline_json = f"""
    {{
        "pipeline": [
            {{
                "type": "readers.las",
                "filename": "{las_path}"
            }}
        ]
    }}
    """

    pipeline = pdal.Pipeline(pipeline_json)
    pipeline.execute()

    arrays = pipeline.arrays
    arr = arrays[0]

    points = np.vstack((arr['X'], arr['Y'], arr['Z'])).T
    intensity = arr['Intensity'].astype(np.float32)

    return points, intensity


In [81]:
def voxelize(points, intensity, voxel_size):
    voxel_idx = np.floor(points / voxel_size).astype(np.int32)

    voxel_dict = {}

    for i, v in enumerate(voxel_idx):
        key = tuple(v)
        if key not in voxel_dict:
            voxel_dict[key] = []
        voxel_dict[key].append(intensity[i])

    return voxel_dict


In [82]:
def compute_reflectivity_field(voxel_dict):
    field = {}

    for key, values in voxel_dict.items():
        vals = np.array(values)
        field[key] = {
            "mean": np.mean(vals),
            "var": np.var(vals),
            "count": len(vals)
        }

    return field


In [83]:
def normalize_reflectivity(field):
    means = np.array([v["mean"] for v in field.values()])
    mu, sigma = means.mean(), means.std() + 1e-6

    for v in field.values():
        v["normalized_reflectivity"] = (v["mean"] - mu) / sigma

    return field


In [84]:
def voxel_features_to_pointcloud(voxel_features, voxel_size):
    """
    Convert voxel reflectivity field to a point cloud for visualization.
    """
    voxel_centers = []
    colors = []

    for key, feat in voxel_features.items():
        center = (np.array(key) + 0.5) * voxel_size
        voxel_centers.append(center)

        # 映射反射率到颜色（红 = 高反射）
        r = np.clip(feat["normalized_reflectivity"], -2, 2)
        r = (r + 2) / 4
        colors.append([r, 0, 1 - r])

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.array(voxel_centers))
    pcd.colors = o3d.utility.Vector3dVector(np.array(colors))

    return pcd


In [85]:
if __name__ == "__main__":
    las_path = "C:/Users/willw/PycharmProjects/Lidar/data/velodyne_points/las/0000000001.las"
    voxel_size = 0.05  # 20 cm

    points, intensity = load_las_as_numpy(las_path)

    voxel_dict = voxelize(points, intensity, voxel_size)
    voxel_features = compute_reflectivity_field(voxel_dict)
    voxel_features = normalize_reflectivity(voxel_features)

    voxel_pcd = voxel_features_to_pointcloud(voxel_features, voxel_size)
    o3d.visualization.draw_geometries([voxel_pcd])
